In [ ]:
import requests
import json
import time
import pandas as pd
from datetime import datetime, timezone, timedelta
import locale


# --- 1. Session加载函数 (无需修改) ---
def load_session_with_cookies(uid, cookies_path="weibo_cookies.json"):
    """
    加载Cookies并根据提供的真实Request Headers构建一个完整的、高仿真的请求头。
    uid: 用户的ID，用于动态生成正确的Referer。
    """
    try:
        with open(cookies_path, "r") as f:
            cookies_list = json.load(f)

        session = requests.Session()
        for cookie in cookies_list:
            session.cookies.set(cookie["name"], cookie["value"])

        xsrf_token = ""
        for cookie in cookies_list:
            if cookie["name"] == "XSRF-TOKEN":
                xsrf_token = cookie["value"]
                break

        headers = {
            "accept": "application/json, text/plain, */*",
            "accept-language": "zh-CN,zh;q=0.9",
            "client-version": "v2.47.120",
            "priority": "u=1, i",
            "referer": f"https://weibo.com/u/{uid}",
            "sec-ch-ua": '"Google Chrome";v="141", "Not?A_Brand";v="8", "Chromium";v="141"',
            "sec-ch-ua-mobile": "?0",
            "sec-ch-ua-platform": '"Windows"',
            "sec-fetch-dest": "empty",
            "sec-fetch-mode": "cors",
            "sec-fetch-site": "same-origin",
            "server-version": "v2025.09.29.1",
            "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36",
            "x-requested-with": "XMLHttpRequest",
            "x-xsrf-token": xsrf_token,
        }

        session.headers.update(headers)

        if not xsrf_token:
            print("警告: 未在Cookies中找到 XSRF-TOKEN，这可能导致请求失败。")

        print("Session、Cookies和高仿真Headers加载成功。")
        return session

    except FileNotFoundError:
        print(f"错误: {cookies_path} 文件未找到。请先运行登录代码获取Cookies。")
        return None
    except Exception as e:
        print(f"加载Session时发生未知错误: {e}")
        return None


In [ ]:
# --- 2. 评论获取函数 (最终版，增加日期和标签) ---
def get_comments(session, post_id, post_comments_count, post_created_at, max_pages=None,flow=0):
    """
    根据微博ID获取评论。
    新增功能：增加评论发布日期列，并根据传入的微博发布日期生成时间差标签。
    
    :param session: requests.Session对象
    :param post_id: 微博的ID
    :param post_comments_count: 微博显示的总评论数
    :param post_created_at: (新增) 微博的发布日期字符串
    :param max_pages: (可选) 限制最多爬取的评论页数
    """

    # 解析传入的原微博发布日期
    page = 1 # 保持 page 计数用于记录页数
    max_id = 0 # 新增：初始化 max_id
    max_id_type = 0 # 保持 max_id_type 初始值
    post_date = post_created_at
    # try:
    #     post_date_aware = datetime.strptime(post_created_at, '%a %b %d %H:%M:%S %z %Y')
    #     post_date = post_date_aware.astimezone(timezone(timedelta(hours=8))).replace(tzinfo=None)
    # except (ValueError, TypeError):
    #     print(f"警告：无法解析原微博 {post_id} 的日期，无法为评论生成时间标签。")
    #     post_date = None

    comments_api = "https://weibo.com/ajax/statuses/buildComments"
    # page = 1
    all_comments_data = []
    seen_comment_ids = set()

    while True:
        if max_pages and page > max_pages:
            print(f"已达到设定的最大页数 {max_pages}，停止获取此微博的评论。")
            break

        if len(all_comments_data) >= post_comments_count:
            print(f"抓取数量已接近或超过帖子总评论数 {post_comments_count}，停止获取。")
            break

        params = {
            "flow": flow,  # 0-热度，1-时间
            "is_reload": 1,
            "id": post_id,
            "is_show_bulletin": 2,
            "is_mix": 0,
            "max_id": max_id,
            "max_id_type": max_id_type
            # "page": page,
        }
        try:
            response = session.get(comments_api, params=params)
            response.raise_for_status()
            data = response.json()

            new_max_id = data.get('max_id', 0)
            new_max_id_type = data.get('max_id_type', 0)


            comments = data.get('data', [])
            if not comments:
                # 如果当前页没有评论数据，则退出
                break

            #    将下一页的 max_id 赋值给当前 max_id
            max_id = new_max_id
            max_id_type = new_max_id_type

            new_comment_found_on_page = False

            for comment in comments:
                # if comment.get('id') != comment.get('rootid'):
                #     continue

                comment_id = comment.get('id')
                if comment_id in seen_comment_ids:
                    continue

                seen_comment_ids.add(comment_id)
                new_comment_found_on_page = True

                user = comment.get('user', {})
                comment_created_at = comment.get('created_at', '')
                time_label = ''

                # 计算时间标签
                if post_date and comment_created_at:
                    try:
                        original_locale = locale.getlocale(locale.LC_TIME)
                        try:
                            locale.setlocale(locale.LC_TIME, 'en_US.UTF-8') 
                        except:
                            pass # Windows下可能报错，忽略
                            
                        dt = datetime.strptime(comment_created_at, '%a %b %d %H:%M:%S %z %Y')
                        locale.setlocale(locale.LC_TIME, original_locale) 
                        comment_date =  pd.to_datetime(dt).tz_localize(None)
                        # comment_date_aware = datetime.strptime(comment_created_at, '%a %b %d %H:%M:%S %z %Y')
                        # comment_date = comment_date_aware.astimezone(timezone(timedelta(hours=8))).replace(tzinfo=None)

                        delta_days = (comment_date.date() - post_date.date()).days

                        if delta_days == 0:
                            time_label = '当日'
                        elif 1 <= delta_days <= 3:
                            time_label = '1-3天'
                        elif 4 <= delta_days <= 7:
                            time_label = '4-7天'
                        elif 8 <= delta_days <= 15:
                            time_label = '8-15天'
                        elif 16 <= delta_days <= 30:
                            time_label = '16-30天'
                        else:
                            time_label = '30天以上'

                    except (ValueError, TypeError):
                        time_label = '日期解析失败'

                comment_bubble = comment.get('comment_bubble', {})
                bubble_text = comment_bubble.get('name', '')

                all_comments_data.append({
                    'post_id': post_id,
                    'post_date': post_date,
                    'comment_content': comment.get('text_raw', ''),
                    'comment_created_at': comment_date, # 新增：评论发出日期
                    'comment_time_label': time_label,       # 新增：时间差标签
                    'comment_likes': comment.get('like_counts', 0),
                    'comment_replies': comment.get('total_number', 0),
                    'user_id': user.get('id'),
                    'user_title': user.get('verified_reason', ''),
                    'fan_level': bubble_text
                })
                
            # 4. 【判断是否退出】 仅在游标不再更新时退出，确保当前页已处理
            if max_id == 0: 
                print("API返回 max_id 为 0，已到达评论末页，停止获取。")
                break
                

            print(f"已获取微博 {post_id} 的第 {page} 页热度评论...")
            page += 1

            if not new_comment_found_on_page:
                print("当前页未发现新评论，可能已到达末页，停止获取。")
                break

            time.sleep(1)
        except Exception as e:
            print(f"获取评论时出错: {e}")
            break

    return all_comments_data

## 主函数

In [ ]:
from datetime import datetime, timezone, timedelta
# --- 3. 微博爬取主函数 (修改调用方式) ---
def scrape_official_account(uid, author,start_date_str, end_date_str=None, comment_max_pages=None, originals_only=False, flow=0):
    """
    爬取指定UID用户在指定日期区间内的微博和评论。
    """
    session = load_session_with_cookies(uid=uid) 
    if not session:
        return

    start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = datetime.strptime(end_date_str, '%Y-%m-%d') if end_date_str else datetime.now()
    end_date = end_date.replace(hour=23, minute=59, second=59)

    print(f"设定爬取区间: 从 {start_date.strftime('%Y-%m-%d')} 到 {end_date.strftime('%Y-%m-%d')}")
    print(f"评论提取模式: 按热度排序，最多提取 {comment_max_pages or '全部'} 页")
    print(f"微博筛选模式: {'仅保留原创微博' if originals_only else '包含原创和转发微博'}")

    posts_api = "https://weibo.com/ajax/statuses/mymblog"
    page = 1
    all_posts_data = []
    all_comments_data = []
    
    stop_scraping = False

    print(f"开始爬取用户 {uid} 的微博...")
    while not stop_scraping:
        params = {'uid': uid, 'page': page, 'feature': 0}
        # try:
        response = session.get(posts_api, params=params)
        response.raise_for_status()
        data = response.json()

        posts = data.get('data', {}).get('list', [])
        if not posts:
            print("已获取所有微博页面，任务结束。")
            break

        print(f"正在处理第 {page} 页的微博...")

        for post in posts:
            created_at_str = post.get('created_at')
            try:
                # post_date_aware = datetime.strptime(created_at_str, '%a %b %d %H:%M:%S %z %Y')
                # post_date = post_date_aware.astimezone(timezone(timedelta(hours=8))).replace(tzinfo=None)
                original_locale = locale.getlocale(locale.LC_TIME)
                try:
                    locale.setlocale(locale.LC_TIME, 'en_US.UTF-8') 
                except:
                    pass # Windows下可能报错，忽略
            
                dt = datetime.strptime(created_at_str, '%a %b %d %H:%M:%S %z %Y')
                locale.setlocale(locale.LC_TIME, original_locale) 
                post_date = pd.to_datetime(dt).tz_localize(None)
            except (ValueError, TypeError):
                print(f"跳过：无法解析日期: {created_at_str}")
                continue

            is_pinned = post.get('isTop') == 1
            
            if not (start_date <= post_date <= end_date):
                if is_pinned:
                    print(f"跳过：置顶微博发布于 {post_date.strftime('%Y-%m-%d')}，不在指定区间内。")
                else:
                    if post_date < start_date:
                        stop_scraping = True
                        print("已遇到早于起始日期的常规微博，停止翻页。")
                continue
            
            is_retweet = 'retweeted_status' in post

            if originals_only and is_retweet:
                print(f"筛选模式: 跳过一条转发微博。")
                continue
            
            print(f"处理：{'转发' if is_retweet else '原创'}微博发布于 {post_date.strftime('%Y-%m-%d')}，在指定区间内。")
            
            post_id = post.get('id')
            post_comments_count = post.get('comments_count', 0)
            
            all_posts_data.append({
                'post_id': post_id,
                'content': post.get('text_raw', ''),
                'reposts_count': post.get('reposts_count', 0),
                'comments_count': post_comments_count,
                'likes_count': post.get('attitudes_count', 0),
                'created_at': post_date,
                'type': 'retweet' if is_retweet else 'original'
            })
            
            # ---  核心改动点 ---
            # 将 post_id, post_comments_count 和 created_at_str 作为独立参数传入
            comments = get_comments(
                session, 
                post_id, 
                post_comments_count, 
                post_date,  # 传入微博发布日期字符串
                max_pages=comment_max_pages,
                flow=flow
            )
            all_comments_data.extend(comments)
        
        if stop_scraping:
            break
        
        page += 1
        time.sleep(2)
        # except Exception as e:
        #     print(f"获取微博列表时出错: {e}")
        #     break

    if not all_posts_data:
        print("在指定日期区间内未找到任何符合条件的微博。")
        return
        
    df_posts = pd.DataFrame(all_posts_data)
    df_comments = pd.DataFrame(all_comments_data)
    return df_posts, df_comments



In [ ]:
# # 获取2025年9月1日至9月30日期间，该账号发布的【原创】微博
# scrape_official_account(
#     uid='7801655101', 
#     start_date_str='2025-09-01', 
#     end_date_str='2025-09-30',
#     originals_only=True  # 设为True，开启仅原创模式
# )

## 无限暖暖

In [ ]:
# 获取2025年9月1日至9月30日期间，该账号发布的【全部】微博
start_date_str='2024-12-04'
end_date_str='2025-12-01'
uid='7801655101'
author = '无限暖暖'

df_posts, df_comments = scrape_official_account(
    uid=uid, 
    author = author,
    start_date_str=start_date_str, 
    end_date_str=end_date_str,
    originals_only=False,# 设为False，或直接省略该参数
    flow=0 #排序方式（0-热度，1-时间）
)



In [ ]:
uid='7801655101'
author = '无限暖暖'
# df_posts['official_uid'] = ""
df_posts['official_uid'] = uid
df_posts['official_author'] = author
df_comments['official_uid'] = uid
df_comments['official_author'] = author

output_path_posts = "C:\\tongji\\0 code\\00_data\\raw_weibo_posts"
output_path_comments = "C:\\tongji\\0 code\\00_data\\raw_weibo_comments"

filename_posts = f"weibo_posts_{author}_{start_date_str}_to_{end_date_str}.csv"
filename_comments = f"weibo_comments_{author}_{start_date_str}_to_{end_date_str}.csv"

df_posts.to_csv(f"{output_path_posts}\\{filename_posts}", index=False, encoding='utf-8-sig')
df_comments.to_csv(f"{output_path_comments}\\{filename_comments}", index=False, encoding='utf-8-sig')
print(f"数据已保存到 {filename_posts} 和 {filename_comments}")

## 无限暖暖搬砖工

In [ ]:
start_date_str='2024-12-04'
end_date_str='2025-12-01'

# start_date_str='2025-11-01'
# end_date_str='2025-11-02'
uid='7915828567'
author = '无限暖暖搬砖工'

df_posts, df_comments = scrape_official_account(
    uid=uid, 
    author = author,
    start_date_str=start_date_str, 
    end_date_str=end_date_str,
    originals_only=False,# 设为False，或直接省略该参数
    flow=0 #排序方式（0-热度，1-时间）
)

In [ ]:
df_posts['official_uid'] = uid
df_posts['official_author'] = author
df_comments['official_uid'] = uid
df_comments['official_author'] = author

output_path_posts = "C:\\tongji\\0 code\\00_data\\raw_weibo_posts"
output_path_comments = "C:\\tongji\\0 code\\00_data\\raw_weibo_comments"

filename_posts = f"weibo_posts_{author}_{start_date_str}_to_{end_date_str}.csv"
filename_comments = f"weibo_comments_{author}_{start_date_str}_to_{end_date_str}.csv"

df_posts.to_csv(f"{output_path_posts}\\{filename_posts}", index=False, encoding='utf-8-sig')
df_comments.to_csv(f"{output_path_comments}\\{filename_comments}", index=False, encoding='utf-8-sig')
print(f"数据已保存到 {filename_posts} 和 {filename_comments}")

## 美鸭梨

In [ ]:
uid='7915670982'
author = '无限暖暖小助手美鸭梨'

df_posts, df_comments = scrape_official_account(
    uid=uid, 
    author = author,
    start_date_str=start_date_str, 
    end_date_str=end_date_str,
    originals_only=False,# 设为False，或直接省略该参数
    flow=0 #排序方式（0-热度，1-时间）
)

In [ ]:
df_posts['official_uid'] = uid
df_posts['official_author'] = author
df_comments['official_uid'] = uid
df_comments['official_author'] = author

output_path_posts = "C:\\tongji\\0 code\\00_data\\raw_weibo_posts"
output_path_comments = "C:\\tongji\\0 code\\00_data\\raw_weibo_comments"

filename_posts = f"weibo_posts_{author}_{start_date_str}_to_{end_date_str}.csv"
filename_comments = f"weibo_comments_{author}_{start_date_str}_to_{end_date_str}.csv"

df_posts.to_csv(f"{output_path_posts}\\{filename_posts}", index=False, encoding='utf-8-sig')
df_comments.to_csv(f"{output_path_comments}\\{filename_comments}", index=False, encoding='utf-8-sig')
print(f"数据已保存到 {filename_posts} 和 {filename_comments}")

In [ ]:
# for i in range(1,10):
#     scrape_official_account(
#         uid='7801655101', 
#         start_date_str=f'2025-{i:02d}-01', 
#         end_date_str=f'2025-{i:02d}-31',
#         originals_only=False,# 设为False，或直接省略该参数
#         flow=0 #排序方式（0-热度，1-时间）
#     )

In [ ]:
# # 获取2025年9月1日至9月30日期间，该账号发布的【全部】微博
# scrape_official_account(
#     uid='6800695256', 
#     start_date_str='2025-10-01', 
#     end_date_str='2025-10-02',
#     originals_only=False,# 设为False，或直接省略该参数
#     flow=0 #排序方式（0-热度，1-时间）
# )